In [42]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
from sklearn.metrics import classification_report

# Set a fixed seed so results are reproducible every time we run the code
torch.manual_seed(42)

# 1. LOAD DATA
# Read the training and validation CSV files into pandas DataFrames
# Each file has two columns: 'text' (the tweet) and 'label' (the sentiment)
train_df = pd.read_csv("sent_train.csv")
valid_df  = pd.read_csv("sent_valid.csv")

print(f"Train: {len(train_df)} samples  |  Valid: {len(valid_df)} samples")
print(train_df["label"].value_counts())

Train: 9543 samples  |  Valid: 2388 samples
label
2    6178
1    1923
0    1442
Name: count, dtype: int64


In [43]:
# 2. TEXT PREPROCESSING
# We clean the tweets before training to remove noise that does not
# help the model learn sentiment (URLs, stock tickers, hashtags, etc.)
def clean(text):
    text = str(text).lower()               # convert to lowercase
    text = re.sub(r"http\S+", "", text)    # remove URLs
    text = re.sub(r"\$\w+", "", text)      # remove stock tickers like $AAPL
    text = re.sub(r"#\w+", "", text)       # remove hashtags like #Finance
    text = re.sub(r"[^a-z\s]", "", text)   # keep only letters and spaces
    return text.strip()

train_df["clean"] = train_df["text"].apply(clean)
valid_df["clean"]  = valid_df["text"].apply(clean)

In [44]:
# 3. BUILD VOCABULARY
# We create a dictionary that maps each word to a unique integer index.
# We only use words from the training set to avoid data leakage.
counter = Counter()
for text in train_df["clean"]:
    counter.update(text.split())

# Keep only the 10,000 most frequent words to limit memory usage
# <PAD> (index 0) is used for padding shorter sequences
# <UNK> (index 1) replaces words not found in the vocabulary
VOCAB_SIZE = 10_000
vocab = ["<PAD>", "<UNK>"] + [w for w, _ in counter.most_common(VOCAB_SIZE - 2)]
w2i   = {w: i for i, w in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)} tokens")

Vocabulary size: 10000 tokens


In [45]:
# 4. ENCODE SEQUENCES WITH PADDING
# Each tweet is converted into a fixed-length list of integers.
# If a tweet is shorter than MAX_LEN, we pad it with zeros at the end.
# If it is longer, we cut it off at MAX_LEN tokens.
MAX_LEN = 50

def encode(text):
    tokens = text.split()[:MAX_LEN]
    ids = [w2i.get(t, 1) for t in tokens]      # unknown words → index 1
    ids += [0] * (MAX_LEN - len(ids))           # pad with zeros if too short
    return ids

train_df["enc"] = train_df["clean"].apply(encode)
valid_df["enc"]  = valid_df["clean"].apply(encode)

In [46]:
# 5. PYTORCH DATASET AND DATALOADER 
# A Dataset object tells PyTorch how to access individual samples.
# A DataLoader groups samples into batches and shuffles the training data.
class TweetDS(Dataset):
    def __init__(self, df):
        self.X = torch.tensor(df["enc"].tolist(),   dtype=torch.long)
        self.y = torch.tensor(df["label"].tolist(), dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(TweetDS(train_df), batch_size=64, shuffle=True)
valid_loader  = DataLoader(TweetDS(valid_df),  batch_size=64)

In [47]:
# 6. LSTM MODEL DEFINITION 
class SentimentLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        # Embedding layer: converts each word index into a dense vector of 128 numbers.
        # padding_idx=0 means the <PAD> token always produces a zero vector.
        self.emb  = nn.Embedding(len(vocab), 128, padding_idx=0)

        # Dropout layer: randomly sets some values to zero during training.
        # This prevents the model from memorising the training data (overfitting).
        self.drop = nn.Dropout(0.3)

        # Bidirectional LSTM with 2 stacked layers and 128 hidden units per direction.
        # 'Bidirectional' means the model reads the tweet from left to right
        # AND from right to left, which helps capture context better.
        self.lstm = nn.LSTM(128, 128, num_layers=2,
                            batch_first=True,
                            bidirectional=True,
                            dropout=0.3)

        # Final linear layer: maps the LSTM output to 3 class scores.
        # We use 128*2 because the LSTM is bidirectional (two directions).
        self.fc = nn.Linear(128 * 2, 3)

    def forward(self, x):
        x = self.drop(self.emb(x))            # (batch, seq_len, 128)
        _, (h, _) = self.lstm(x)              # h shape: (num_layers*2, batch, 128)

        # Take the last hidden state from both directions and concatenate them.
        # h[-2] = forward direction, h[-1] = backward direction
        h = torch.cat([h[-2], h[-1]], dim=1)  # (batch, 256)

        return self.fc(self.drop(h))          # (batch, 3) — one score per class

device = "cpu"
model  = SentimentLSTM().to(device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Total parameters: 1,940,227


In [48]:
# 7. LOSS FUNCTION AND OPTIMIZER 
# CrossEntropyLoss is the standard loss for multi-class classification.
# It measures how far the model's predictions are from the true labels.
criterion = torch.optim.Adam and nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss()

# Adam is an adaptive optimizer that adjusts the learning rate automatically.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ReduceLROnPlateau reduces the learning rate by half if the validation loss
# does not improve for 2 consecutive epochs, helping the model converge better.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

best_acc = 0

In [49]:
# 8. TRAINING LOOP 
for epoch in range(10):

    # Training phase
    # model.train() activates dropout and other training-only behaviours
    model.train()
    train_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()          # reset gradients from the previous step
        loss = criterion(model(X), y)  # compute the loss
        loss.backward()                # backpropagation: calculate gradients

        # Gradient clipping prevents gradients from becoming too large, which is a common problem in RNN-based models like LSTMs
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()               # update the model weights
        train_loss += loss.item()

    # Validation phase
    # model.eval() disables dropout so predictions are deterministic
    model.eval()
    correct, total, val_loss = 0, 0, 0
    with torch.no_grad():              # no need to compute gradients during evaluation
        for X, y in valid_loader:
            X, y = X.to(device), y.to(device)
            out  = model(X)
            val_loss += criterion(out, y).item()
            correct  += (out.argmax(1) == y).sum().item()
            total    += len(y)

    acc = correct / total
    scheduler.step(val_loss)

    # Save the model weights whenever we reach a new best validation accuracy
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_model.pt")

    print(f"Epoch {epoch+1:>2}/10 | Train Loss: {train_loss/len(train_loader):.4f} "
          f"| Val Loss: {val_loss/len(valid_loader):.4f} | Val Acc: {acc:.4f}")

Epoch  1/10 | Train Loss: 0.7962 | Val Loss: 0.6652 | Val Acc: 0.7404
Epoch  2/10 | Train Loss: 0.6404 | Val Loss: 0.5921 | Val Acc: 0.7693
Epoch  3/10 | Train Loss: 0.5341 | Val Loss: 0.5499 | Val Acc: 0.7940
Epoch  4/10 | Train Loss: 0.4504 | Val Loss: 0.5660 | Val Acc: 0.8095
Epoch  5/10 | Train Loss: 0.3671 | Val Loss: 0.5318 | Val Acc: 0.8132
Epoch  6/10 | Train Loss: 0.3026 | Val Loss: 0.5490 | Val Acc: 0.8065
Epoch  7/10 | Train Loss: 0.2643 | Val Loss: 0.5540 | Val Acc: 0.8137
Epoch  8/10 | Train Loss: 0.2244 | Val Loss: 0.6116 | Val Acc: 0.8120
Epoch  9/10 | Train Loss: 0.1740 | Val Loss: 0.6539 | Val Acc: 0.8107
Epoch 10/10 | Train Loss: 0.1540 | Val Loss: 0.6724 | Val Acc: 0.8149
